# Реальная лабораторная 05 — Air Quality

Сенсорные измерения, закодированные пропуски и временной дрейф.

> **Как работать:** выполняй ячейки последовательно. Сначала сформулируй ожидание, затем запускай код и объясняй результат. Не удаляй строки и не исправляй значения автоматически до выяснения их смысла.

**Данные:** скачиваются из UCI Machine Learning Repository через `download_uci_datasets.py`. Первый запуск требует интернета; после сохранения CSV лабораторная выполняется локально.

## 0. Исследовательский контракт

Запиши единицу наблюдения, целевую аудиторию анализа, момент принятия решения и то, какие выводы допустимы. Разделяй описательное исследование, прогноз и причинные утверждения.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

SEED = 42
RNG = np.random.default_rng(SEED)
DATA_DIR = Path("data/raw")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 60)

def load_real(slug):
    path = DATA_DIR / f"{slug}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Не найден {path}. Сначала выполните: "
            f"python download_uci_datasets.py --only {slug}"
        )
    return pd.read_csv(path, low_memory=False)

def section(title, text):
    display(Markdown(f"### {title}\n\n{text}"))

## 1. Загрузка и проверка кодирования

In [ ]:
air = load_real("air_quality")
print(air.shape); display(air.head()); display(air.dtypes.to_frame("dtype"))

## 2. Sentinel values и качество датчиков

In [ ]:
num_cols = air.select_dtypes(include=np.number).columns
sentinel_report = pd.DataFrame({
    "n_minus_200": [(air[c] == -200).sum() for c in num_cols],
    "pct_minus_200": [100*(air[c] == -200).mean() for c in num_cols],
    "n_missing": [air[c].isna().sum() for c in num_cols]
}, index=num_cols).sort_values("pct_minus_200",ascending=False)
display(sentinel_report)
air_clean = air.copy()
air_clean[num_cols] = air_clean[num_cols].replace(-200, np.nan)
print("NaN count after sentinel conversion:", int(air_clean.isna().sum().sum()))

## 3. Временная ось и дрейф

In [ ]:
date_col = next((c for c in air_clean.columns if c.lower()=="date"), None)
time_col = next((c for c in air_clean.columns if c.lower()=="time"), None)
if date_col and time_col:
    date_raw = air_clean[date_col].astype("string").str.strip()
    time_raw = air_clean[time_col].astype("string").str.strip().str.replace(".", ":", regex=False)
    date_part = pd.to_datetime(date_raw, dayfirst=True, errors="coerce")
    time_part = pd.to_timedelta(time_raw, errors="coerce")
    dt = date_part + time_part
    # Do not silently drop observations: keep invalid timestamps visible in the audit.
    air_clean["timestamp"] = dt
    valid_time = air_clean.dropna(subset=["timestamp"]).copy()
    print("Invalid timestamps:", air_clean["timestamp"].isna().sum())
    if not valid_time.empty:
        valid_time = valid_time.sort_values("timestamp")
        sensor_cols=[c for c in valid_time.select_dtypes(include=np.number).columns]
        candidate=next((c for c in sensor_cols if "PT08.S1" in c or "CO(GT)" in c), None)
        if candidate:
            fig,ax=plt.subplots(figsize=(13,4)); valid_time.set_index("timestamp")[candidate].plot(ax=ax); ax.set_title(candidate+" over time"); plt.show()
else:
    print("UCI API may have returned Date/Time combined or transformed; inspect columns and construct the timestamp explicitly.")

## 4. Вопросы к сенсорному исследованию

In [ ]:
missing_by_col = air_clean.isna().mean().sort_values(ascending=False)
display(missing_by_col.to_frame("missing_share"))
if "timestamp" in air_clean and air_clean["timestamp"].notna().any():
    time_indexed = air_clean.dropna(subset=["timestamp"]).set_index("timestamp")
    month_missing = time_indexed.isna().resample("MS").mean()
    display(month_missing.head())
else:
    print("Timestamp parsing failed for this representation. Inspect raw Date/Time values and document the conversion before temporal analysis.")

## Самостоятельные задания

1. Сравни долю sentinel missing по каждому датчику. Почему единый процент пропусков недостаточен?
2. Проверь временную концентрацию пропусков. Можно ли считать их MCAR?
3. Сравни распределения до и после корректного преобразования -200 → NaN.
4. Исследуй синхронность и cross-sensitivity сенсоров.
5. Найди возможные признаки sensor drift и предложи тест устойчивости по временным блокам.

## Итоговый отчёт

В финальной версии notebook должны быть: **вопрос → единица анализа → происхождение и ограничения данных → проверка качества → описательная статистика → закономерности → неопределённость → риски смещения/утечки → выводы → следующие проверки**. Каждый вывод должен ссылаться на таблицу, график или расчёт.